In [ ]:
# --- 1. INSTALAÇÃO ---
print("Instalando bibliotecas...")
!pip install geopandas folium shapely gspread google-auth google-auth-oauthlib google-auth-httplib2 ipynbname unidecode fuzzywuzzy python-Levenshtein -q
print("Bibliotecas instaladas.")

# --- 2. IMPORTAÇÕES ---
import os
import time
import datetime
import math
import requests
import zipfile
import pandas as pd
import geopandas as gpd
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from io import BytesIO
from shapely.geometry import Point, mapping
from shapely.ops import polygonize
from unidecode import unidecode
from fuzzywuzzy import fuzz
import folium
from branca.element import MacroElement
from jinja2 import Template
from google.colab import userdata, drive, auth as colab_auth
from google.auth import default as google_auth_default
import gspread
import urllib3
import numpy as np
import warnings
from IPython.display import display

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# --- 3. CONFIGURAÇÃO ---
try:
    drive.mount('/content/drive')
    base_path_secret = userdata.get('DRIVE_SAVE_PATH') if 'DRIVE_SAVE_PATH' in userdata else "/content/drive/MyDrive/Colab_Pipelines/Maia_Bus_Audit/"
    DRIVE_SAVE_PATH = base_path_secret
    if not os.path.exists(DRIVE_SAVE_PATH): os.makedirs(DRIVE_SAVE_PATH)
except:
    DRIVE_SAVE_PATH = "/content/"

# Secrets & Email Config
try:
    GMAIL_USER = userdata.get('GMAIL_USER')
    GMAIL_APP_PASSWORD = userdata.get('GMAIL_APP_PASSWORD')
    email_list_raw = userdata.get('DESTINATION_EMAIL_LIST')
    if "," in email_list_raw:
        DESTINATION_EMAIL_LIST = [e.strip() for e in email_list_raw.split(',')]
    else:
        DESTINATION_EMAIL_LIST = [email_list_raw.strip()]
except Exception as e:
    print(f"⚠️ Aviso: Secrets de email não configurados corretamente. ({e})")
    GMAIL_USER = None
    DESTINATION_EMAIL_LIST = []

# URLs
MAIA_BOUNDARY_URL = "https://baze.cm-maia.pt/BaZe/api/api4gj.php?nome=limconcvm"
GTFS_URL = "https://opendata.porto.digital/dataset/5275c986-592c-43f5-8f87-aabbd4e4f3a4/resource/89a6854f-2ea3-4ba0-8d2f-6558a9df2a98/download/horarios_gtfs_stcp_16_04_2025.zip"

HTML_MAP_FILENAME = os.path.join(DRIVE_SAVE_PATH, f"auditoria_stcp_osm_{datetime.date.today().isoformat()}.html")
GOOGLE_SHEET_NAME = f"Auditoria_STCP_OSM_{datetime.date.today().isoformat()}"

# Servidores OSM
OVERPASS_URLS = [
    "https://overpass.kumi.systems/api/interpreter",
    "https://lz4.overpass-api.de/api/interpreter",
    "https://overpass-api.de/api/interpreter"
]
API_TIMEOUT = 180
MATCH_THRESHOLD = 100 # Raio de busca largo para apanhar erros grandes
DIST_LIMIT_STRICT = 25.0 # Metros para considerar "No sítio certo"

# --- ESTILOS VISUAIS ---
STYLE_CONFIG = {
    "Total": {"folium": "green", "hex": "green", "icon": "check", "label": "Match Confirmado (<25m)"},

    "ConflitoLoc": { "folium": "orange", "hex": "#FFA500", "icon": "arrows-alt", "label": "Conflito Localização (>25m)" },
    "ConflitoNome": { "folium": "blue", "hex": "#1E90FF", "icon": "font", "label": "Conflito de Nomes" },
    "ConflitoMisto": { "folium": "red", "hex": "#FF0000", "icon": "bomb", "label": "Conflito Misto (Nome + Local)" },

    "Apenas": {"folium": "purple", "hex": "purple", "icon": "bus", "label": "Apenas no GTFS (Falta no Mapa)"},
    "Default": {"folium": "gray", "hex": "gray", "icon": "info", "label": "Indefinido"}
}

# --- 4. FUNÇÕES AUXILIARES & PROFILING ---
profiling_log = []
script_start_time = time.time()

def start_timer(): return time.time()

def log_time(task, start_t):
    elapsed_proc = time.time() - start_t
    elapsed_watch = time.time() - script_start_time
    print(f"[{task}] {elapsed_proc:.2f}s")
    profiling_log.append({"task": task, "proc_time": elapsed_proc, "watch_time": elapsed_watch})

def normalize_text(text):
    if not text: return ""
    return unidecode(str(text)).strip().lower()

def approx_distance_meters(lat1, lon1, lat2, lon2):
    try:
        if not all([lat1, lon1, lat2, lon2]): return float('inf')
        R = 6371000
        phi1, phi2 = math.radians(lat1), math.radians(lat2)
        dphi = math.radians(lat2 - lat1)
        dlambda = math.radians(lon2 - lon1)
        a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2)**2
        return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
    except: return float('inf')

def get_safe_bounds(poly):
    try:
        minx, miny, maxx, maxy = poly.bounds
        if np.isnan(minx): raise ValueError("Bounds NaN")
        return minx, miny, maxx, maxy
    except:
        return -8.691, 41.185, -8.530, 41.309

context_vars = {'sheet_url': '#', 'date': str(datetime.date.today())}

# --- 5. CARREGAMENTO DE DADOS ---

def load_maia_boundary(url):
    s = start_timer()
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            r = requests.get(url, verify=False, timeout=30)
            gdf = gpd.read_file(BytesIO(r.content))
            if gdf.empty: return None
            if gdf.crs and gdf.crs.to_string() != "EPSG:4326": gdf = gdf.to_crs("EPSG:4326")

            geom = gdf.geometry.unary_union
            if geom.geom_type in ['LineString', 'MultiLineString']:
                polys = list(polygonize(geom))
                if polys: geom = max(polys, key=lambda a: a.area)
                else: geom = geom.convex_hull
            elif not geom.is_valid:
                geom = geom.buffer(0)

            log_time("Limite Maia", s)
            return geom
    except Exception as e:
        print(f"Erro limite: {e}")
        return None

def load_gtfs_stcp(poly_geom):
    s = start_timer()
    print("A baixar GTFS STCP...")
    try:
        r = requests.get(GTFS_URL, verify=False, timeout=60)
        with zipfile.ZipFile(BytesIO(r.content)) as z:
            if 'stops.txt' not in z.namelist(): return gpd.GeoDataFrame()
            with z.open('stops.txt') as f:
                df = pd.read_csv(f, dtype={'stop_lat': float, 'stop_lon': float})

        geometry = [Point(xy) for xy in zip(df.stop_lon, df.stop_lat)]
        gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")
        gdf['Name'] = gdf['stop_name']
        gdf['Name_Clean'] = gdf['Name'].apply(normalize_text)

        filtered = gdf[gdf.within(poly_geom)].copy()
        log_time(f"GTFS STCP ({len(filtered)} paragens)", s)
        return filtered
    except Exception as e:
        print(f"Erro GTFS: {e}")
        return gpd.GeoDataFrame()

def query_osm_bus_bulletproof(polygon, timeout_seconds, buffer, urls_list):
    s = start_timer()
    print("A carregar OSM...")
    minx, miny, maxx, maxy = get_safe_bounds(polygon)
    bbox = f"{miny-buffer:.5f},{minx-buffer:.5f},{maxy+buffer:.5f},{maxx+buffer:.5f}"

    query = f"""
    [out:json][timeout:{timeout_seconds}];
    (
      node["highway"="bus_stop"]({bbox});
      way["highway"="bus_stop"]({bbox});
      node["public_transport"="platform"]({bbox});
      way["public_transport"="platform"]({bbox});
      node["amenity"="bus_station"]({bbox});
    );
    out center tags;
    """
    for url in urls_list:
        try:
            r = requests.post(url, data={"data": query}, timeout=timeout_seconds)
            if r.status_code == 200:
                data = r.json().get("elements", [])
                log_time(f"OSM Raw ({len(data)})", s)
                return data
            elif r.status_code == 429: time.sleep(2)
        except: continue
    print("❌ Falha Total no OSM.")
    return []

def process_osm_elements(elements):
    features = []
    if not elements: return gpd.GeoDataFrame()
    for e in elements:
        lat, lon = None, None
        if e['type'] == 'node': lat, lon = e.get("lat"), e.get("lon")
        elif "center" in e: lat, lon = e["center"]["lat"], e["center"]["lon"]
        if lat and lon:
            tags = e.get("tags", {})
            name = tags.get("name", "Sem Nome")
            features.append({
                "Name": name, "Name_Clean": normalize_text(name),
                "Latitude": lat, "Longitude": lon,
                "OSM_ID": e.get("id"), "geometry": Point(lon, lat)
            })
    return gpd.GeoDataFrame(features, crs="EPSG:4326")

def filter_points_safe(gdf, poly):
    if gdf.empty: return gdf
    if gdf.crs != "EPSG:4326": gdf = gdf.to_crs("EPSG:4326")
    try:
        filtered = gdf[gdf.within(poly)].copy()
        return filtered
    except: return gdf

# --- 7. RECONCILIAÇÃO (GTFS vs OSM) ---
def reconcile_bus(gtfs, osm):
    s = start_timer()
    print("A reconciliar (GTFS vs OSM)...")
    results = []

    def find_best_match(target, candidates):
        if candidates.empty: return None, None, None, None
        df = candidates.copy()

        df['dist'] = df.geometry.apply(lambda g: approx_distance_meters(target.geometry.y, target.geometry.x, g.y, g.x))


        candidates_near = df[df['dist'] <= MATCH_THRESHOLD].copy()
        if candidates_near.empty: return None, None, None, None


        best_match = candidates_near.sort_values('dist').iloc[0]

        comment = None
        raw_dist = best_match['dist']
        raw_name = best_match['Name']

        gtfs_clean = target['Name_Clean']
        match_clean = best_match['Name_Clean']
        name_score = fuzz.token_set_ratio(gtfs_clean, match_clean)

        if raw_dist > DIST_LIMIT_STRICT:
            comment = f"Mudou de local? (Dist: {int(raw_dist)}m)"

        if name_score < 70 and len(match_clean) > 3 and "sem nome" not in match_clean:
            add_txt = f"Nome difere: '{raw_name}'"
            comment = f"{comment} | {add_txt}" if comment else add_txt

        return raw_name, raw_dist, best_match.geometry, comment

    for _, row in gtfs.iterrows():
        o_name, o_dist, o_geom, o_comm = find_best_match(row, osm)

        if o_name:
            has_loc_issue = o_dist > DIST_LIMIT_STRICT
            has_name_issue = "Nome difere" in (o_comm or "")

            if has_loc_issue and has_name_issue:
                code, status = "ConflitoMisto", "⚠️ Conflito Misto"
            elif has_loc_issue:
                code, status = "ConflitoLoc", "⚠️ Conflito de Localização"
            elif has_name_issue:
                code, status = "ConflitoNome", "⚠️ Conflito de Nomes"
            else:
                code, status = "Total", "Match Confirmado"
        else:
            code, status = "Apenas", "Apenas no GTFS"

        def gc(geo): return (geo.y, geo.x) if geo else (None, None)
        def fmt(val): return f"{round(val, 2)}" if val is not None else "-"

        results.append({
            'Nome_GTFS': row['Name'], 'ID_GTFS': row['stop_id'],
            'Status': status, 'Code': code, 'Comentarios': o_comm or "",
            'Nome_OSM': o_name or 'N/A', 'Dist_OSM': fmt(o_dist),
            'Lat_GTFS': row.geometry.y, 'Lon_GTFS': row.geometry.x,
            'Lat_OSM': gc(o_geom)[0], 'Lon_OSM': gc(o_geom)[1]
        })

    log_time("Reconciliação Bus", s)
    return pd.DataFrame(results)

# --- 8. MAPA ---
class MapLegend(MacroElement):
    _template = Template(u"""
    {% macro html(this, kwargs) %}
    <div style="position: fixed; bottom: 30px; right: 30px; width: 340px; z-index:9999; border: 2px solid rgba(0,0,0,0.2); border-radius: 8px; background-color: rgba(255, 255, 255, 0.95); padding: 15px; font-family: 'Segoe UI', Arial, sans-serif; font-size: 13px; box-shadow: 0 4px 6px rgba(0,0,0,0.1);">

    <h4 style="margin-top:0; margin-bottom: 12px; font-size: 16px; color: #333; border-bottom: 1px solid #ddd; padding-bottom: 8px;"><b>{{ this.title }}</b></h4>

    <table style="width:100%; border-collapse: collapse;">
        {% for label, config in this.items %}
        <tr style="margin-bottom: 8px; border-bottom: 1px solid #f0f0f0;">
            <td style="width: 30px; text-align: center; padding: 6px;"><i class="fa fa-{{ config.icon }}" style="color: {{ config.hex }}; font-size: 16px;"></i></td>
            <td style="padding: 6px; color: #444;">{{ label }}</td>
        </tr>
        {% endfor %}
    </table>

    <div style="margin-top: 10px; border-top: 1px solid #eee; padding-top: 8px;">
        <small style="color:#555; font-weight:bold;">Linha de Desvio:</small>
        <div style="font-size: 11px; margin-top:4px;">
            <span style="color:blue; font-weight:bold; font-size: 14px;">---</span> Diferença de Local (>25m)
        </div>
    </div>

    <div style="margin-top: 10px; border-top: 1px solid #eee; padding-top: 8px; text-align: center;">
        <small style="color:#555; font-weight:bold; display:block; margin-bottom:4px;">Fontes de Dados</small>
        <div style="font-size: 12px; font-weight: bold; color: #444;">
            <a href="https://www.stcp.pt/" target="_blank" style="color:#0078A8; text-decoration:none;">GTFS STCP</a>
            <span style="color:#ccc; margin:0 4px;">|</span>
            <a href="https://www.openstreetmap.org/" target="_blank" style="color:#0078A8; text-decoration:none;">OSM</a>
        </div>
    </div>

    </div>
    {% endmacro %}
    """)
    def __init__(self, title, items):
        super(MapLegend, self).__init__()
        self._name = 'MapLegend'
        self.title = title
        self.items = items

def create_map_bus(gdf, poly_geom, filename):
    print("Gerando Mapa...")
    m = folium.Map(location=[41.23, -8.62], zoom_start=13, tiles="CartoDB positron")
    if poly_geom is not None:
        try:
            folium.GeoJson(data=mapping(poly_geom.simplify(0.0001)), name="Limite Maia", style_function=lambda x: {'color': '#000000', 'weight': 3, 'fillOpacity': 0, 'dashArray': '5, 5'}).add_to(m)
        except: pass

    def is_valid(lat, lon):
        try:
            if lat is None or lon is None: return False
            if math.isnan(float(lat)) or math.isnan(float(lon)): return False
            return True
        except: return False

    if not gdf.empty:
        for _, r in gdf.iterrows():
            if not is_valid(r['Lat_GTFS'], r['Lon_GTFS']): continue
            code = r['Code']
            config = STYLE_CONFIG.get(code, STYLE_CONFIG['Default'])
            comment_html = f"<div style='margin-top:5px; padding: 6px; background-color: #ffebee; border: 1px solid #ffcdd2; color: #b71c1c; font-size: 11px;'>⚠️ {r['Comentarios']}</div>" if r['Comentarios'] else ""

            popup_html = f"""
            <div style='font-family: "Segoe UI", sans-serif; width: 320px; font-size: 13px;'>
                <div style='background-color: {config['hex']}; color: {'white' if 'Conflito' in code else 'black'}; padding: 6px 10px; border-radius: 4px;'><b>{r['Status']}</b></div>
                {comment_html}
                <div style="margin-top: 8px;"><b style='color:#333; font-size:15px;'>🚌 {r['Nome_GTFS']}</b> <span style="font-size:11px; color:#777;">(ID: {r['ID_GTFS']})</span></div>
                <div style='margin-top:8px; padding: 8px; background-color: #f0f7fb; border-left: 3px solid #0078A8;'><b>📍 GTFS:</b> {r['Lat_GTFS']:.5f}, {r['Lon_GTFS']:.5f}</div>
                <hr>
                <div style="margin-top:5px;">
                   <b>🌍 OSM:</b> {r['Nome_OSM']} ({r['Dist_OSM']}m)
                </div>
            </div>"""

            folium.Marker([r['Lat_GTFS'], r['Lon_GTFS']], popup=folium.Popup(popup_html, max_width=350), icon=folium.Icon(color=config['folium'], icon=config['icon'], prefix='fa')).add_to(m)

            # Desenha linha de erro
            if is_valid(r['Lat_OSM'], r['Lon_OSM']) and float(r['Dist_OSM']) > DIST_LIMIT_STRICT:
                folium.PolyLine([[r['Lat_GTFS'], r['Lon_GTFS']], [r['Lat_OSM'], r['Lon_OSM']]], color="blue", weight=2, dash_array="5,5").add_to(m)

    items = [(v['label'], v) for k, v in STYLE_CONFIG.items() if k != "Default"]
    m.add_child(MapLegend("Auditoria STCP vs OSM", items))
    m.save(filename)
    return m

def save_sheet(df, name):
    try:
        colab_auth.authenticate_user(); creds, _ = google_auth_default(); gc = gspread.authorize(creds)
        try: sh = gc.open(name)
        except: sh = gc.create(name)
        ws = sh.get_worksheet(0); ws.clear()
        df_save = df.drop(columns=['geometry'], errors='ignore')
        ws.update([df_save.columns.values.tolist()] + df_save.fillna('').astype(str).values.tolist(), 'A1')
        for e in DESTINATION_EMAIL_LIST:
            try: sh.share(e, perm_type='user', role='writer')
            except: pass
        return f"https://docs.google.com/spreadsheets/d/{sh.id}"
    except: return "#"

# --- 9. FUNÇÃO DE EMAIL ---
def send_execution_email(sheet_url):
    if not GMAIL_USER or not DESTINATION_EMAIL_LIST:
        print("⏭️ Email não enviado (Credenciais em falta).")
        return

    print("📧 A enviar email de relatório...")

    log_rows = ""
    for entry in profiling_log:
        log_rows += f"""
        <tr>
            <td style="border: 1px solid black; padding: 4px; font-family: monospace; white-space: nowrap;">{entry['task']}</td>
            <td style="border: 1px solid black; padding: 4px; text-align: right; font-family: monospace;">{entry['watch_time']:.2f}</td>
            <td style="border: 1px solid black; padding: 4px; text-align: right; font-family: monospace;">{entry['proc_time']:.2f}</td>
        </tr>
        """

    total_time = time.time() - script_start_time

    html_body = f"""
    <html>
    <body style="font-family: Arial, sans-serif;">
        <h2 style="color: #2E86C1;">Pipeline Bus Audit - Execution Log</h2>
        <p>O processo de auditoria de paragens de AUTOCARRO (GTFS vs OSM) foi concluído.</p>

        <p><b>🔗 Google Sheet Resultado:</b> <a href="{sheet_url}">{sheet_url}</a></p>

        <h3 style="margin-bottom: 5px; margin-top:20px;">Execution Timing Log</h3>

        <table style="border-collapse: collapse; width: 100%; font-size: 12px; border: 2px solid black; font-family: Arial, sans-serif;">
            <thead style="background-color: #f2f2f2;">
                <tr>
                    <th style="border: 1px solid black; padding: 6px; text-align:left;">Task(s)</th>
                    <th style="border: 1px solid black; padding: 6px; text-align: right;">watch time (secs)</th>
                    <th style="border: 1px solid black; padding: 6px; text-align: right;">proc time (secs)</th>
                </tr>
            </thead>
            <tbody>
                {log_rows}
            </tbody>
            <tfoot>
                <tr style="font-weight: bold; background-color: #e6e6e6;">
                    <td style="border: 1px solid black; padding: 6px;">Overall (before email):</td>
                    <td style="border: 1px solid black; padding: 6px; text-align: right;">{total_time:.2f}</td>
                    <td style="border: 1px solid black; padding: 6px; text-align: right;">{total_time:.2f}</td>
                </tr>
            </tfoot>
        </table>

        <p style="font-size: 11px; color: #777; margin-top: 20px;">Automated via Google Colab Pipeline.</p>
    </body>
    </html>
    """

    msg = MIMEMultipart()
    msg['From'] = GMAIL_USER
    msg['To'] = ", ".join(DESTINATION_EMAIL_LIST)
    msg['Subject'] = f"Pipeline Log: Maia STCP Audit {datetime.date.today()}"
    msg.attach(MIMEText(html_body, 'html'))

    try:
        server = smtplib.SMTP_SSL('smtp.gmail.com', 465)
        server.login(GMAIL_USER, GMAIL_APP_PASSWORD)
        server.sendmail(GMAIL_USER, DESTINATION_EMAIL_LIST, msg.as_string())
        server.quit()
        print("✅ Email enviado com sucesso!")
    except Exception as e:
        print(f"❌ Erro ao enviar email: {e}")

# --- EXECUÇÃO ---
print("\n--- INICIO PIPELINE AUTOCARROS ---")
maia_geom = load_maia_boundary(MAIA_BOUNDARY_URL)

if maia_geom is not None:
    gtfs = load_gtfs_stcp(maia_geom)
    if not gtfs.empty:
        osm_elements = query_osm_bus_bulletproof(maia_geom, API_TIMEOUT, 0.01, OVERPASS_URLS)
        osm = process_osm_elements(osm_elements)
        if not osm.empty: osm = filter_points_safe(osm, maia_geom)

        df_res = reconcile_bus(gtfs, osm)

        if not df_res.empty:
            url = save_sheet(df_res, GOOGLE_SHEET_NAME)
            context_vars['sheet_url'] = url
            m = create_map_bus(df_res, maia_geom, HTML_MAP_FILENAME)
            print(f"✅ Mapa gerado: {HTML_MAP_FILENAME}")

            send_execution_email(url)
        else: print("❌ Falha na reconciliação.")
    else: print("❌ Nenhuma paragem STCP encontrada.")
else: print("❌ Erro no Limite da Maia.")

Instalando bibliotecas...
Bibliotecas instaladas.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

--- INICIO PIPELINE AUTOCARROS ---
[Limite Maia] 2.15s
A baixar GTFS STCP...
[GTFS STCP (338 paragens)] 1.51s
A carregar OSM...
[OSM Raw (2848)] 5.74s
A reconciliar (GTFS vs OSM)...
[Reconciliação Bus] 13.04s
Gerando Mapa...
✅ Mapa gerado: /content/auditoria_stcp_osm_2026-01-22.html
📧 A enviar email de relatório...
✅ Email enviado com sucesso!


In [ ]:
display(m)